In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

# Cấu hình hiển thị tiếng Việt cho biểu đồ (nếu cần)
plt.rcParams['font.family'] = 'Arial' 

# 1. Kết nối tới database
def get_db_connection():
    return mysql.connector.connect(
        host="localhost",
        user="root",        # Thay bằng user của bạn
        password="", # Thay bằng password của bạn
        database="datathon26"
    )

conn = get_db_connection()

# ---------------------------------------------------------
# CẤP ĐỘ 1: DESCRIPTIVE (Thống kê: Doanh thu theo thời gian)
# ---------------------------------------------------------
query_sales = """
SELECT o.order_date, SUM(p.payment_value) as daily_revenue
FROM orders o
JOIN payments p ON o.order_id = p.order_id
GROUP BY o.order_date
ORDER BY o.order_date
"""
df_sales = pd.read_sql(query_sales, conn)
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

plt.figure(figsize=(12, 5))
sns.lineplot(data=df_sales, x='order_date', y='daily_revenue', color='#2ecc71', linewidth=2)
plt.title('BIỂU ĐỒ XU HƯỚNG DOANH THU THEO NGÀY', fontsize=14)
plt.xlabel('Ngày đặt hàng')
plt.ylabel('Doanh thu (USD)')
plt.grid(alpha=0.3)
plt.show()

# ---------------------------------------------------------
# CẤP ĐỘ 2: DIAGNOSTIC (Giải thích: Tại sao có đơn hoàn?)
# ---------------------------------------------------------
query_returns = """
SELECT o.order_source, COUNT(o.order_id) as total, COUNT(r.order_id) as returned
FROM orders o
LEFT JOIN returns r ON o.order_id = r.order_id
GROUP BY o.order_source
"""
df_returns = pd.read_sql(query_returns, conn)
df_returns['return_rate'] = (df_returns['returned'] / df_returns['total']) * 100

plt.figure(figsize=(10, 6))
sns.barplot(data=df_returns.sort_values('return_rate'), x='order_source', y='return_rate', palette='magma')
plt.title('TỶ LỆ HOÀN HÀNG THEO NGUỒN ĐƠN (DIAGNOSTIC)', fontsize=14)
plt.ylabel('Tỷ lệ hoàn (%)')
plt.show()

# ---------------------------------------------------------
# CẤP ĐỘ 3 & 4: PREDICTIVE & PRESCRIPTIVE (Đề xuất: Hiệu quả Khuyến mãi)
# ---------------------------------------------------------
query_promo = """
SELECT pr.promo_id, SUM(p.payment_value) as revenue, COUNT(DISTINCT o.order_id) as orders
FROM promotions pr
JOIN order_items oi ON pr.promo_id = oi.promo_id
JOIN orders o ON oi.order_id = o.order_id
JOIN payments p ON o.order_id = p.order_id
GROUP BY pr.promo_id
"""
df_promo = pd.read_sql(query_promo, conn)
df_promo['AOV'] = df_promo['revenue'] / df_promo['orders'] # Giá trị đơn hàng trung bình

plt.figure(figsize=(10, 6))
scatter = sns.scatterplot(data=df_promo, x='orders', y='revenue', size='AOV', hue='promo_id', sizes=(100, 1000))
plt.title('MA TRẬN HIỆU QUẢ KHUYẾN MÃI (PRESCRIPTIVE)', fontsize=14)
plt.xlabel('Số lượng đơn hàng')
plt.ylabel('Tổng doanh thu mang lại')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

conn.close()

/var/folders/tn/l785m_3s5bg6mvddm1v1pb_c0000gn/T/ipykernel_19401/3811527346.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sales = pd.read_sql(query_sales, conn)


ValueError: time data "apple_pay" doesn't match format "%Y-%m-%d". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.